In [1]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [2]:
TRAINING_JSON = Path('./../data/raw/training.json')
TEST_JSON = Path('./../data/raw/test.json')
IMAGES_DIR = Path('./../data/raw/images')

In [3]:
with open(TRAINING_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(TEST_JSON, "r", encoding="utf-8") as f:
    data1 = json.load(f)

print(len(data))
print(len(data1))

1208
120


In [4]:
raw_path = data[0]["image"]["pathname"]
fname = Path(raw_path).name
fname

'8d02117d-6c71-4e47-b50a-6cc8d5eb1d55.png'

In [5]:
item = data[0]

print(json.dumps(item, indent=4, ensure_ascii=False))

{
    "image": {
        "checksum": "676bb8e86fc2dbf05dd97d51a64ac0af",
        "pathname": "/images/8d02117d-6c71-4e47-b50a-6cc8d5eb1d55.png",
        "shape": {
            "r": 1200,
            "c": 1600,
            "channels": 3
        }
    },
    "objects": [
        {
            "bounding_box": {
                "minimum": {
                    "r": 1057,
                    "c": 1440
                },
                "maximum": {
                    "r": 1158,
                    "c": 1540
                }
            },
            "category": "red blood cell"
        },
        {
            "bounding_box": {
                "minimum": {
                    "r": 868,
                    "c": 1303
                },
                "maximum": {
                    "r": 971,
                    "c": 1403
                }
            },
            "category": "red blood cell"
        },
        {
            "bounding_box": {
                "minimum": {
               

In [6]:
# ---------------------------------------------------------
# 1. Load annotations in the REAL JSON format
# ---------------------------------------------------------

def load_detection_annotations(json_path):
    """
    Reads the JSON in the provided format (image + objects + bounding boxes)
    and returns a structured list for later use.
    
    Returned structure:
    [
        {
            'image': 'file_name.png',
            'width': int,
            'height': int,
            'objects': [
                {
                    'bbox': [xmin, ymin, xmax, ymax],
                    'category': string
                },
                ...
            ]
        },
        ...
    ]
    """

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    samples = []

    for item in data:

        raw_path = item["image"]["pathname"]
        fname = Path(raw_path).name

        width = item["image"]["shape"]["c"]
        height = item["image"]["shape"]["r"]

        objs = []
        for obj in item["objects"]:

            bb = obj["bounding_box"]
            ymin = bb["minimum"]["r"]
            xmin = bb["minimum"]["c"]
            ymax = bb["maximum"]["r"]
            xmax = bb["maximum"]["c"]

            objs.append({
                "bbox": [xmin, ymin, xmax, ymax],   # format xmin, ymin, xmax, ymax
                "category": obj["category"]
            })

        samples.append({
            "image": fname,
            "width": width,
            "height": height,
            "objects": objs
        })

    return samples


# ---------------------------------------------------------
# 2. Load image + RGB + numpy
# ---------------------------------------------------------
def load_image(path, as_rgb=True, to_numpy=True):
    img = Image.open(path)

    if as_rgb:
        img = img.convert("RGB")

    if to_numpy:
        return np.array(img)

    return img


# ---------------------------------------------------------
# 3. Build dataset: images + annotations + bounding boxes
# ---------------------------------------------------------
def build_dataset(json_path, images_dir, max_items=None):
    """
    Returns a list of tuples:
    (image_numpy, annotations, real_image_path)

    where annotations =
    {
        'objects': [
            {'bbox': [...], 'category': 'trophozoite'},
            ...
        ],
        'width': W,
        'height': H
    }
    """
    annotations = load_detection_annotations(json_path)
    dataset = []
    missing = []

    for i, entry in enumerate(annotations):

        if max_items and i >= max_items:
            break

        fname = entry["image"]
        img_path = images_dir / fname

        # If not found, try basename only
        if not img_path.exists():
            alt = images_dir / Path(fname).name
            if alt.exists():
                img_path = alt
            else:
                missing.append(fname)
                continue

        try:
            img_arr = load_image(img_path)
        except Exception as e:
            print(f"Error opening {img_path}: {e}")
            continue

        dataset.append((img_arr, entry, str(img_path)))

    print(f"Total read: {len(dataset)}, missing: {len(missing)}")

    if missing:
        print("Missing files (examples):", missing[:10])

    return dataset

In [7]:
dataset = build_dataset(TRAINING_JSON, IMAGES_DIR)

Total read: 1208, missing: 0


In [8]:
dataset[0]

(array([[[ 85, 101, 122],
         [ 85, 101, 122],
         [ 87, 103, 125],
         ...,
         [238, 240, 239],
         [241, 243, 242],
         [244, 246, 245]],
 
        [[ 85, 100, 122],
         [ 88, 103, 125],
         [ 90, 105, 127],
         ...,
         [238, 240, 239],
         [241, 243, 242],
         [244, 246, 245]],
 
        [[ 86, 100, 122],
         [ 88, 103, 125],
         [ 93, 108, 130],
         ...,
         [238, 240, 239],
         [241, 243, 242],
         [244, 246, 245]],
 
        ...,
 
        [[ 52,  66,  98],
         [ 55,  69, 102],
         [ 58,  72, 104],
         ...,
         [ 74,  78, 113],
         [ 74,  77, 110],
         [ 76,  79, 113]],
 
        [[ 47,  62,  97],
         [ 51,  65, 101],
         [ 53,  67, 102],
         ...,
         [ 75,  81, 115],
         [ 78,  82, 115],
         [ 80,  84, 118]],
 
        [[ 40,  55,  90],
         [ 43,  57,  94],
         [ 46,  60,  96],
         ...,
         [ 75,  81, 115],
  

In [10]:
# list all images

# for i, (img, label, fname) in enumerate(dataset):
    
#     print(f"Image {i}: {fname} (label = {label})")

#     plt.imshow(img)
#     plt.axis("off")
#     plt.show()

In [11]:
def extract_classes(dataset):
    classes = set()
    for _, entry, _ in dataset:
        for obj in entry["objects"]:
            classes.add(obj["category"])
    return sorted(list(classes))

In [12]:
import tensorflow as tf
import numpy as np

def tf_dataset_from_samples(samples, class_to_id, img_size=(512, 512)):

    def gen():
        for img, ann, _ in samples:
            h, w, _ = img.shape

            boxes = []
            labels = []

            for obj in ann["objects"]:
                xmin, ymin, xmax, ymax = obj["bbox"]
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(class_to_id[obj["category"]])

            yield img.astype("float32"), {
                "boxes": np.array(boxes, dtype="float32"),
                "labels": np.array(labels, dtype="int32"),
                "image_hw": (h, w)
            }

    output_signature = (
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.float32),
        {
            "boxes": tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
            "labels": tf.TensorSpec(shape=(None,), dtype=tf.int32),
            "image_hw": tf.TensorSpec(shape=(2,), dtype=tf.int32),
        }
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    # preprocessing pipeline
    def preprocess(img, ann):
        img = img / 255.0
        img = tf.image.resize(img, img_size)

        # ajustar as bounding boxes para o novo tamanho
        orig_h, orig_w = ann["image_hw"][0], ann["image_hw"][1]
        scale_x = img_size[1] / tf.cast(orig_w, tf.float32)
        scale_y = img_size[0] / tf.cast(orig_h, tf.float32)

        boxes = ann["boxes"]
        boxes = tf.stack([
            boxes[:, 0] * scale_x,
            boxes[:, 1] * scale_y,
            boxes[:, 2] * scale_x,
            boxes[:, 3] * scale_y
        ], axis=1)

        ann["boxes"] = boxes
        return img, ann

    return ds.map(preprocess)


2025-11-24 13:03:40.707632: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [14]:
!pip install keras-cv

  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Preparing metadata (setup.py) ... done
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 1.2 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 1.3 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 840.7 kB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 399.3 kB/s eta 0:00:000:00:010:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 1.2 MB/s eta 0:00:001.2 MB/s eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 3.1 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (807 kB)
Using cach

In [15]:
import keras_cv
import tensorflow as tf

def build_retinanet(num_classes, img_size=(512, 512)):

    model = keras_cv.models.RetinaNet(
        classes=num_classes,
        bounding_box_format="xyxy",
        backbone=keras_cv.models.ResNet50Backbone(
            include_rescaling=False
        )
    )

    # compilação recomendada
    model.compile(
        classification_loss="focal",
        box_loss="smooth_l1",
        optimizer=tf.keras.optimizers.Adam(1e-4),
        jit_compile=True
    )

    return model

/home/balda/Documentos/conformalized-xplainable-malaria/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
samples = build_dataset(TRAINING_JSON, IMAGES_DIR)
class_names = extract_classes(samples)

class_to_id = {c: i for i, c in enumerate(class_names)}
id_to_class = {v: k for k, v in class_to_id.items()}

train_ds = tf_dataset_from_samples(samples, class_to_id, img_size=(512,512))
train_ds = train_ds.shuffle(1000).batch(4).prefetch(tf.data.AUTOTUNE)

model = build_retinanet(num_classes=len(class_names))

model.fit(train_ds, epochs=20)
